# Test d'un AI Agent sur Google Colab

Ce notebook montre comment construire et tester un **agent IA simple** capable d'utiliser des outils (tool use / function calling) avec l'API Claude d'Anthropic.

Étapes :
1. Installer les dépendances
2. Configurer la clé API
3. Définir un outil (calculatrice)
4. Implémenter la boucle de l'agent
5. Tester l'agent avec quelques questions

In [ ]:
!pip install -q anthropic

## Configuration de la clé API

Renseigne ta clé API Anthropic ci-dessous (obtenue sur https://console.anthropic.com/).

Astuce Colab : tu peux aussi stocker la clé dans les *Secrets* de Colab (icône clé dans la barre latérale) sous le nom `ANTHROPIC_API_KEY` puis la charger avec `google.colab.userdata`.

In [ ]:
import os
from getpass import getpass

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass

if not api_key:
    api_key = getpass('Entre ta cle API Anthropic: ')

os.environ['ANTHROPIC_API_KEY'] = api_key

## Definition des outils

On donne a l'agent deux outils :
- `calculator` : outil **cote client**, qu'on implemente et executons nous-memes.
- `web_search` : outil **serveur** integre a l'API Claude — Anthropic execute la recherche web lui-meme, aucun code cote client requis.

In [ ]:
import ast
import operator as op

# Evaluateur d'expressions arithmetiques securise (pas d'eval() brut)
_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
    ast.Div: op.truediv, ast.Pow: op.pow, ast.USub: op.neg,
}

def _eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp):
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError('Expression non supportee')

def calculator(expression: str) -> str:
    try:
        result = _eval(ast.parse(expression, mode='eval').body)
        return str(result)
    except Exception as e:
        return f'Erreur: {e}'

tools = [
    {
        'name': 'calculator',
        'description': 'Evalue une expression arithmetique simple (+, -, *, /, **) et retourne le resultat.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'expression': {
                    'type': 'string',
                    'description': "L'expression mathematique a evaluer, ex: '12 * (3 + 4)'",
                }
            },
            'required': ['expression'],
        },
    },
    {
        # Outil de recherche web integre a l'API Claude (outil "serveur") :
        # Anthropic execute la recherche lui-meme, aucune implementation cote client requise.
        'type': 'web_search_20260209',
        'name': 'web_search',
        'max_uses': 5,
    },
]

## Boucle de l'agent

L'agent envoie le message au modele, execute les outils demandes, renvoie les resultats, et repete jusqu'a obtenir une reponse finale.

In [ ]:
import anthropic

client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'

TOOL_FUNCTIONS = {
    'calculator': calculator,
}

def run_agent(user_message: str, max_turns: int = 5) -> str:
    messages = [{'role': 'user', 'content': user_message}]

    for _ in range(max_turns):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        messages.append({'role': 'assistant', 'content': response.content})

        if response.stop_reason != 'tool_use':
            return ''.join(
                block.text for block in response.content if block.type == 'text'
            )

        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                func = TOOL_FUNCTIONS.get(block.name)
                output = func(**block.input) if func else f'Outil inconnu: {block.name}'
                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': str(output),
                })

        messages.append({'role': 'user', 'content': tool_results})

    return "Nombre maximum d'iterations atteint sans reponse finale."

## Tests de l'agent

In [ ]:
print(run_agent('Combien font 234 * 18 + 7 ?'))

In [ ]:
print(run_agent("Explique-moi en une phrase ce qu'est un agent IA, puis calcule 2**10."))

In [ ]:
print(run_agent("Cherche sur le web qui a remporte le dernier prix Nobel de physique et resume la reponse en une phrase."))